# Étape 3 — Générer tous les audios du catalogue

**Projet ɖevi ƒe agbalẽ · Hack4Children 2026**

Ce notebook lit **`catalogue_messages.xlsx`** et **`prononciation.json`**, puis génère d'un coup l'audio de chaque message, avec les mêmes règles que le notebook 01 (dictionnaire, pauses, vitesse 1,0).

- Message **sans variable** → un seul audio : `messages/bienvenue.wav`
- Message avec **`{prenom}`** → un audio par **prénom du dictionnaire** : `messages/croissance__Kosi.wav`
- Message avec **`{prenom}` et `{centre}`** → un audio par prénom **et** par centre : `messages/rappel_veille__Kosi__sylvanus_olympio.wav`
- Le notebook écrit aussi **`index_messages.json`**, le fichier que lit l'application.
- Chaque audio est aussi converti en **`.ogg` (Opus)**, le format des messages vocaux WhatsApp.

Quand tu modifies un texte dans Excel : tu relances ce notebook, c'est tout. Ne recopie pas les messages dans le notebook 01.

**Avant de lancer :** dans le panneau *Fichiers* de Colab, dépose `catalogue_messages.xlsx` et `prononciation.json` (ou utilise la cellule 1).

In [ ]:
# 1. Déposer les deux fichiers (ignorer si déjà déposés dans /content/)
import os
from google.colab import files
if not (os.path.exists("/content/catalogue_messages.xlsx") and os.path.exists("/content/prononciation.json")):
    files.upload()
print(sorted(f for f in os.listdir("/content") if f.endswith((".xlsx", ".json"))))

In [ ]:
!pip install -q transformers scipy openpyxl

In [ ]:
# 2. Modèle et réglages
import torch, unicodedata, re, json, subprocess, shutil
import numpy as np, pandas as pd, scipy.io.wavfile as wavfile
from IPython.display import Audio, display
from transformers import VitsModel, AutoTokenizer, set_seed

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("facebook/mms-tts-ewe")
mod = VitsModel.from_pretrained("facebook/mms-tts-ewe").to(device).eval()
SR = mod.config.sampling_rate
vocab = set(tok.get_vocab().keys())

P = json.load(open("/content/prononciation.json", encoding="utf-8"))
PRONONCIATION = P["prononciation"]
REGLAGES = P["reglages"]
CENTRES = P.get("centres", {})
print("Réglages :", REGLAGES)
print("Centres :", {k: v["voix"] for k, v in CENTRES.items()})

In [ ]:
# 3. Règles du texte (identiques au notebook 01)
REMPLACEMENTS = {"õ": "õ", "ê": "e", "ð": "ɖ"}
PONCTUATION = set(".,!?;:")

def nettoyer(texte):
    """Retire guillemets et espaces en trop, ajoute l'espace manquant après un point."""
    t = str(texte or "")
    t = re.sub(r"[\"“”«»]", "", t)
    t = re.sub(r"([.!?;:,])(?=[^\s.!?;:,])", r"\1 ", t)
    t = re.sub(r"\s+([,.!?;:])", r"\1", t)
    return re.sub(r"\s+", " ", t).strip()

def normaliser(texte):
    t = unicodedata.normalize("NFC", texte.lower())
    for a, b in REMPLACEMENTS.items():
        t = t.replace(a, b)
    return t

def appliquer_prononciation(texte):
    t = normaliser(texte)
    for mot, forme in sorted(PRONONCIATION.items(), key=lambda kv: -len(kv[0])):
        t = re.sub(rf"(?<!\w){re.escape(normaliser(mot))}(?!\w)", normaliser(forme), t)
    return t

def problemes(texte):
    return sorted({c for c in normaliser(texte) if c not in vocab and not c.isspace() and c not in PONCTUATION})

def decouper(texte):
    m = re.split(r"([.!?;:]+|,)", texte)
    out = []
    for i in range(0, len(m), 2):
        bout, signe = m[i].strip(), (m[i + 1] if i + 1 < len(m) else "")
        if bout:
            pause = REGLAGES["pause_virgule_s"] if signe == "," else (REGLAGES["pause_phrase_s"] if signe else 0.0)
            out.append((bout, pause))
    return out

def voix(texte):
    t = appliquer_prononciation(texte)
    if problemes(t):
        raise ValueError(f"caractères non lus : {problemes(t)}")
    parties = []
    for bout, pause in decouper(t):
        set_seed(REGLAGES["graine"])
        mod.speaking_rate, mod.noise_scale = REGLAGES["vitesse"], REGLAGES["bruit"]
        with torch.no_grad():
            parties.append(mod(**tok(bout, return_tensors="pt").to(device)).waveform[0].cpu().numpy())
        if pause:
            parties.append(np.zeros(int(pause * SR), dtype=np.float32))
    return np.concatenate(parties), t

## 4. Prénoms et centres

Tous les prénoms de `prononciation.json` sont générés. Les orthographes qui se prononcent pareil (Kosi / Kossi) partagent le même audio.
Pour ajouter un prénom : l'ajouter dans `prononciation.json` (orthographe → forme pour la voix), puis relancer.

In [ ]:
NON_PRENOMS = {"vaksin", "Sylvanus Olympio"}   # entrées du dictionnaire qui ne sont pas des prénoms

PRENOMS = {}   # forme voix -> orthographes acceptées (la première sert de nom de fichier)
for ecrit, forme in PRONONCIATION.items():
    if ecrit not in NON_PRENOMS:
        PRENOMS.setdefault(forme, []).append(ecrit)
GROUPES = [{"cle": v[0], "orthographes": v} for v in PRENOMS.values()]
DUREE_MAX = 15   # secondes : au-delà, le message est signalé comme long
for g in GROUPES:
    print(f"{g['cle']:10s} <- {g['orthographes']}")
print(len(GROUPES), "prénoms ;", len(CENTRES), "centres")

In [ ]:
# 5. Lecture du catalogue
cat = pd.read_excel("/content/catalogue_messages.xlsx", sheet_name="Messages").fillna("")
cat = cat[cat["id"].astype(str).str.strip() != ""]
cat["texte_voix"] = [nettoyer(v if str(v).strip() else e) for v, e in zip(cat["éwé pour la voix"], cat["éwé écrit"])]
print(len(cat), "messages ;", (cat["texte_voix"] == "").sum(), "sans texte")
cat[["id", "priorité", "texte_voix"]]

In [ ]:
# 6. Génération + écoute
SORTIE = "/content/messages"
shutil.rmtree(SORTIE, ignore_errors=True); os.makedirs(SORTIE)

def variantes(texte):
    """[(suffixe, texte, prenom_cle, orthographes, centre)] selon les variables présentes."""
    a_prenom, a_centre = "{prenom}" in texte, "{centre}" in texte
    groupes = GROUPES if a_prenom else [None]
    centres = list(CENTRES) if a_centre else [None]
    out = []
    for g in groupes:
        for c in centres:
            t, suffixe = texte, ""
            if g:
                t = t.replace("{prenom}", g["cle"]); suffixe += f"__{g['cle']}"
            if c:
                t = t.replace("{centre}", CENTRES[c]["voix"]); suffixe += f"__{c}"
            out.append((suffixe, t, g["cle"] if g else None, g["orthographes"] if g else [], c))
    return out

ECOUTER_TOUT = False   # True pour afficher un lecteur pour chacun des audios
rapport = []
for _, r in cat.sort_values("priorité").iterrows():
    print("=" * 70 + f"\n[{r['id']}]  priorité {r['priorité']}  |  {r['français']}")
    if not r["texte_voix"]:
        print("   (pas de texte éwé)"); continue
    restes = re.findall(r"\{[^}]*\}", r["texte_voix"])
    inconnues = [v for v in restes if v not in ("{prenom}", "{centre}")]
    if inconnues:
        print("   variable inconnue :", inconnues)
    for suffixe, texte, prenom, orthographes, centre in variantes(r["texte_voix"]):
        nom = f"{r['id']}{suffixe}"
        try:
            onde, lu = voix(texte)
        except ValueError as err:
            print(f"   {nom} : NON GÉNÉRÉ - {err}")
            rapport.append({"fichier": nom, "id": r["id"], "prenom": prenom, "orthographes": orthographes, "centre": centre,
                            "duree_s": None, "texte_lu": texte, "alerte": str(err)})
            continue
        wav = f"{SORTIE}/{nom}.wav"
        wavfile.write(wav, SR, (np.clip(onde, -1, 1) * 32767).astype(np.int16))
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav, "-c:a", "libopus", "-b:a", "24k", wav[:-4] + ".ogg"], check=True)
        duree = len(onde) / SR
        alerte = f"long (> {DUREE_MAX} s)" if duree > DUREE_MAX else ""
        rapport.append({"fichier": nom, "id": r["id"], "prenom": prenom, "orthographes": orthographes, "centre": centre,
                        "duree_s": round(duree, 1), "texte_lu": lu, "alerte": alerte})
        print(f"   {nom}  -  {duree:.1f} s  {alerte}")
        if ECOUTER_TOUT or not prenom or prenom == GROUPES[0]["cle"]:
            display(Audio(onde, rate=SR))   # par défaut : un seul prénom affiché par message

rapport = pd.DataFrame(rapport)
rapport.drop(columns=["orthographes"]).to_csv(f"{SORTIE}/rapport_generation.csv", index=False, encoding="utf-8-sig")
print(f"\n{len(rapport)} audios ; {rapport['alerte'].astype(bool).sum()} alertes")
rapport[rapport["alerte"] != ""]

In [ ]:
# 7. index_messages.json : le fichier que lit l'application
BOUTONS = {"manque_3j": [{"bouton": "1", "libelle": "1", "message_suivant": "reponse_vient"},
                         {"bouton": "2", "libelle": "2", "message_suivant": "reponse_empechement"}]}
messages = []
for _, r in cat.iterrows():
    audios = [{"prenom": a["prenom"], "orthographes": a["orthographes"], "centre": a["centre"],
               "ogg": f"voix/audio/messages/{a['fichier']}.ogg", "wav": f"voix/audio/messages/{a['fichier']}.wav",
               "duree_s": a["duree_s"]}
              for _, a in rapport[(rapport["id"] == r["id"]) & rapport["duree_s"].notna()].iterrows()]
    modele = nettoyer(r["éwé écrit"])
    messages.append({"id": r["id"], "categorie": r["catégorie"], "priorite": int(r["priorité"]),
                     "declencheur": r["quand"], "destinataire": r["destinataire"], "canal": r["canal"],
                     "texte_fr": r["français"], "modele_ewe": modele,
                     "variables": sorted(set(re.findall(r"\{(\w+)\}", modele))),
                     "boutons": BOUTONS.get(r["id"], []), "audios": audios})
index = {"version": pd.Timestamp.now().strftime("%Y-%m-%d"), "langue": "ewe",
         "format_whatsapp": "audio/ogg; codecs=opus",
         "prenoms_disponibles": {o: g["cle"] for g in GROUPES for o in g["orthographes"]},
         "centres": {k: v["nom_affiche"] for k, v in CENTRES.items()},
         "messages": messages}
json.dump(index, open(f"{SORTIE}/index_messages.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(len(messages), "messages ;", sum(len(m["audios"]) for m in messages), "audios dans l'index")

## 8. Télécharger

Le `.zip` contient pour chaque message un `.wav` (écoute, journal des messages) et un `.ogg` (envoi WhatsApp), plus `rapport_generation.csv` et `index_messages.json`.
**Vide d'abord** `voix/audio/messages/` (les noms de fichiers ont changé), puis décompresse le `.zip` dedans.

Ensuite, dans `catalogue_messages.xlsx` : passe à **« Validé (audio OK) »** seulement les messages dont tu as écouté et approuvé l'audio ; les autres restent **« Traduit »**.

In [ ]:
files.download(shutil.make_archive("/content/audios_messages", "zip", SORTIE))